# MTA Demand Model Comparison — MLP vs Blend vs LSTM

So sánh ba kiến trúc dự báo **residual log-demand** (demand thực − baseline median theo route×hour×weekend) trên cùng hold-out mùa:

| Model | Mô tả |
|-------|--------|
| **MLP** | Mạng fully-connected với route embedding + BatchNorm; Huber loss trên residual. |
| **Blend** | Kết hợp MLP + HistGradientBoosting; trọng số blend và `RESID_CLIP` tune trên val (`summer_2025`). |
| **LSTM** | Chuỗi 24 giờ (theo route) trên feature đã scale; 2-layer LSTM theo ITM 2026 MTA paper. |

Hold-out: **test = autumn_2025**, **val = summer_2025** (giống `mta_schedule_optimization.ipynb`).
Outputs: `outputs/model_comparison/`


In [1]:
import os
import json
import time
import warnings
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

import sys
import importlib

_cwd = Path(".").resolve()
if (_cwd / "lib" / "single_route_pipeline.py").is_file():
    NOTEBOOK_DIR = _cwd
elif (_cwd / "notebooks" / "lib" / "single_route_pipeline.py").is_file():
    NOTEBOOK_DIR = _cwd / "notebooks"
else:
    raise ImportError(f"Cannot locate notebooks/lib from cwd={_cwd}")

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import lib.single_route_pipeline as srp
importlib.reload(srp)

REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR = REPO_ROOT / "datasets"
SCHEDULE_DIR = REPO_ROOT / "datasets" / "schedule_current"
OUT_DIR = Path("outputs/model_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = DATA_DIR / "train_manifest.json"
train_manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8")) if MANIFEST_FILE.exists() else {}

_DEFAULT_PATHS = {
    "ridership": "ridership.csv",
    "routes_by_station_complex": "routes_by_station_complex.csv",
    "factors_hourly": "factors_hourly.csv",
}

def data_path(key: str) -> Path:
    paths_cfg = train_manifest.get("paths", {}) or {}
    name = paths_cfg.get(key, _DEFAULT_PATHS.get(key, key))
    return DATA_DIR / name

RIDERSHIP_FILE = data_path("ridership")
ROUTES_BY_COMPLEX_FILE = data_path("routes_by_station_complex")
FACTORS_HOURLY_FILE = data_path("factors_hourly")
FACTORS_JOIN_KEYS = ["date", "hour"]

_scope = train_manifest.get("scope", {})
OPT_HOURS_FALLBACK = _scope.get("hours", list(range(24)))
OPT_HOURS = list(OPT_HOURS_FALLBACK)
USE_GTFS_OPERATING_HOURS = True
MIN_GTFS_TRIPS_PER_HOUR = 1
OPT_ROUTE_MODE = _scope.get("route_mode", "headway")
DIRECTIONS = _scope.get("directions", [0, 1])
MIN_ROUTE_DEMAND_ROWS = 50
ROUTE_ALIASES = train_manifest.get("route_aliases", {"SIR": "SI"})
HEADWAY_SERVICE = "Weekday"
COVERAGE_THR = 0.7
CHUNK_ROWS = 1_000_000

HOLDOUT_TEST_SEASON = "autumn"
HOLDOUT_TEST_YEAR = 2025
HOLDOUT_VAL_SEASON = "summer"
HOLDOUT_VAL_YEAR = 2025

USE_ROUTE_EMBEDDING = True
USE_LAG_FEATURES = True
LAG_FEATURE_COLS = list(srp.DEFAULT_LAG_COLS)
PEAK_HOURS = (7, 8, 9, 17, 18, 19)

HOURLY_FACTOR_COLS = [
    "temperature_c", "apparent_temperature_c",
    "precipitation_mm", "rain_mm", "snowfall_cm",
    "windspeed_kmh", "windgusts_kmh",
    "is_rain", "is_snow", "is_severe_wind",
    "is_major_event_window",
]

NN_HIDDEN = (64, 32)
NN_DROPOUT = 0.25
USE_BATCH_NORM = True
NN_EPOCHS_MAIN = 200
PEAK_SAMPLE_WEIGHT = 2.0
RESID_CLIP = (-0.4, 0.4)
RESID_CLIP_CANDIDATES = [0.35, 0.45, 0.55]
TUNE_RESID_CLIP_ON_VAL = True

LSTM_SEQ_LEN = 24
LSTM_HIDDEN = 64
LSTM_LAYERS = 2
LSTM_EPOCHS = NN_EPOCHS_MAIN

print("TensorFlow:", tf.__version__)
print("Data dir:", DATA_DIR.resolve())
print("Output dir:", OUT_DIR.resolve())
for label, p in [
    ("ridership", RIDERSHIP_FILE),
    ("routes_by_station_complex", ROUTES_BY_COMPLEX_FILE),
    ("factors_hourly", FACTORS_HOURLY_FILE),
    ("schedule", SCHEDULE_DIR),
]:
    print(f"  {label}: {'OK' if p.exists() else 'MISSING'} — {p.name if p.is_file() else p}")



TensorFlow: 2.20.0
Data dir: D:\tranport-public\datasets
Output dir: D:\tranport-public\notebooks\outputs\model_comparison
  ridership: OK — ridership.csv
  routes_by_station_complex: OK — routes_by_station_complex.csv
  factors_hourly: OK — factors_hourly.csv
  schedule: OK — D:\tranport-public\datasets\schedule_current


## Load & merge data


In [2]:
# Station -> route map
routes_complex = pd.read_csv(ROUTES_BY_COMPLEX_FILE)
routes_complex["station_complex_id"] = routes_complex["station_complex_id"].astype(str)
route_col = "gtfs_route_ids" if "gtfs_route_ids" in routes_complex.columns else "route_ids_gtfs"
station_to_routes = (
    routes_complex.assign(route=routes_complex[route_col].astype(str).str.split())
    .explode("route")
    .dropna(subset=["route"])
)
station_to_routes = station_to_routes[station_to_routes["route"] != ""]
station_to_routes = station_to_routes[["station_complex_id", "route"]].drop_duplicates()
station_to_routes["route"] = station_to_routes["route"].replace(ROUTE_ALIASES)
valid_stations = set(station_to_routes["station_complex_id"].unique())

# Ridership chunked load
usecols = ["transit_timestamp", "transit_mode", "station_complex_id", "ridership"]
agg_parts = []
for chunk in pd.read_csv(
    RIDERSHIP_FILE,
    usecols=usecols,
    chunksize=CHUNK_ROWS,
    parse_dates=["transit_timestamp"],
    dtype={"station_complex_id": str},
):
    chunk = chunk[chunk["transit_mode"] == "subway"]
    chunk = chunk[chunk["station_complex_id"].isin(valid_stations)]
    if chunk.empty:
        continue
    chunk["date"] = chunk["transit_timestamp"].dt.date
    chunk["hour"] = chunk["transit_timestamp"].dt.hour
    grp = chunk.groupby(["station_complex_id", "date", "hour"], as_index=False)["ridership"].sum()
    agg_parts.append(grp)

ridership_station = pd.concat(agg_parts, ignore_index=True)
ridership_station = (
    ridership_station.groupby(["station_complex_id", "date", "hour"], as_index=False)["ridership"].sum()
)

# GTFS headway -> route-level ridership
headway = srp.build_headway_from_gtfs(SCHEDULE_DIR, service_id=HEADWAY_SERVICE)
headway["route_id"] = headway["route_id"].astype(str)
route_hour_trips = srp.build_route_hour_trip_counts(headway, min_trips=MIN_GTFS_TRIPS_PER_HOUR)
station_route_weights = srp.build_station_route_hour_weights(
    station_to_routes,
    route_hour_trips,
    min_trips=MIN_GTFS_TRIPS_PER_HOUR,
    route_aliases=ROUTE_ALIASES,
)
ridership_route = srp.aggregate_ridership_to_routes(
    ridership_station,
    station_route_weights,
    coverage_thr=COVERAGE_THR,
)

# Factors hourly
def load_factors_hourly(path: Path) -> pd.DataFrame:
    fh = pd.read_csv(path, parse_dates=["timestamp", "date"])
    if "hour" not in fh.columns:
        fh["hour"] = fh["timestamp"].dt.hour
    keep = [
        "date", "hour", "day_of_week", "is_weekend", "is_us_holiday", "month",
        "temperature_c", "apparent_temperature_c",
        "precipitation_mm", "rain_mm", "snowfall_cm",
        "windspeed_kmh", "windgusts_kmh",
        "is_rain", "is_snow", "is_severe_wind",
        "is_major_event_window",
    ]
    keep = [c for c in keep if c in fh.columns]
    fh = fh[keep].drop_duplicates(subset=["date", "hour"]).copy()
    fh["date"] = pd.to_datetime(fh["date"]).dt.normalize()
    fh["hour"] = fh["hour"].astype(int)
    return fh

factors = load_factors_hourly(FACTORS_HOURLY_FILE)

ridership_route["date"] = pd.to_datetime(ridership_route["date"]).dt.normalize()
ridership_route["hour"] = ridership_route["hour"].astype(int)
ridership_all = ridership_route.merge(factors, on=FACTORS_JOIN_KEYS, how="left")

if "day_of_week" in ridership_all.columns:
    ridership_all["day_of_week"] = ridership_all["day_of_week"].fillna(ridership_all["date"].dt.dayofweek)
else:
    ridership_all["day_of_week"] = ridership_all["date"].dt.dayofweek
if "is_weekend" in ridership_all.columns:
    ridership_all["is_weekend"] = ridership_all["is_weekend"].fillna((ridership_all["date"].dt.dayofweek >= 5).astype(int))
else:
    ridership_all["is_weekend"] = (ridership_all["date"].dt.dayofweek >= 5).astype(int)

flag_cols = [
    "is_us_holiday", "is_rain", "is_snow", "is_severe_wind",
    "is_major_event_window",
]
num_cols = [c for c in HOURLY_FACTOR_COLS if c not in flag_cols]
for c in num_cols:
    if c not in ridership_all.columns:
        ridership_all[c] = 0.0
    ridership_all[c] = ridership_all[c].fillna(ridership_all[c].median())
for c in flag_cols:
    if c not in ridership_all.columns:
        ridership_all[c] = 0
    ridership_all[c] = ridership_all[c].fillna(0).astype(int)

# OPT_ROUTES filter
HEADWAY_ROUTES = sorted(headway["route_id"].astype(str).unique())
route_row_counts = ridership_all.groupby("route_id").size()
RIDERSHIP_ROUTES = route_row_counts[route_row_counts >= MIN_ROUTE_DEMAND_ROWS].index.astype(str).tolist()
if OPT_ROUTE_MODE == "headway":
    OPT_ROUTES = HEADWAY_ROUTES
elif OPT_ROUTE_MODE == "intersection":
    OPT_ROUTES = sorted(set(HEADWAY_ROUTES) & set(RIDERSHIP_ROUTES))
else:
    OPT_ROUTES = RIDERSHIP_ROUTES
OPT_ROUTES = [r for r in OPT_ROUTES if r in set(RIDERSHIP_ROUTES)]

route_hours = {r: list(OPT_HOURS_FALLBACK) for r in OPT_ROUTES}
route_dir_hours: dict[tuple[str, int], list[int]] = {}
if USE_GTFS_OPERATING_HOURS:
    route_dir_hours = srp.derive_operating_hours_by_route_direction(
        headway, OPT_ROUTES, directions=DIRECTIONS, min_trips=MIN_GTFS_TRIPS_PER_HOUR
    )
    route_hours = srp.derive_operating_hours_by_route(
        headway, OPT_ROUTES, directions=DIRECTIONS, min_trips=MIN_GTFS_TRIPS_PER_HOUR
    )
    OPT_HOURS = sorted({h for hs in route_dir_hours.values() for h in hs})

df = ridership_all[ridership_all["route_id"].isin(OPT_ROUTES)].copy()
if USE_GTFS_OPERATING_HOURS:
    df = df[df.apply(lambda row: row["hour"] in route_hours.get(row["route_id"], []), axis=1)].copy()

print(f"OPT_ROUTES ({len(OPT_ROUTES)}): {OPT_ROUTES[:8]} ...")
print(f"Dataset: {len(df):,} rows | {df['date'].nunique()} days | demand {df['demand'].min():.0f}–{df['demand'].max():.0f}")



OPT_ROUTES (25): ['1', '2', '3', '4', '5', '6', '7', 'A'] ...
Dataset: 399,840 rows | 731 days | demand 1–47212


## Feature engineering & splits


In [3]:
df = df.sort_values(["route_id", "date", "hour"]).reset_index(drop=True)
if "month" not in df.columns:
    df["month"] = df["date"].dt.month

df = srp.add_cyclical_time_features(df)

route_to_idx = {r: i for i, r in enumerate(sorted(OPT_ROUTES))}
df["route_idx"] = df["route_id"].map(route_to_idx).astype(int)
df = srp.add_lag_features(df, use_lags=USE_LAG_FEATURES, lag_cols=LAG_FEATURE_COLS)

unique_dates = sorted(df["date"].unique())
train_dates, val_dates, test_dates, holdout_meta = srp.build_seasonal_holdout_splits(
    unique_dates,
    test_season=HOLDOUT_TEST_SEASON,
    test_season_year=HOLDOUT_TEST_YEAR,
    val_season=HOLDOUT_VAL_SEASON,
    val_season_year=HOLDOUT_VAL_YEAR,
    auto_adjust=False,
    return_meta=True,
)
HOLDOUT_TEST_SEASON = holdout_meta["test_season"]
HOLDOUT_TEST_YEAR = holdout_meta["test_season_year"]
HOLDOUT_VAL_SEASON = holdout_meta["val_season"]
HOLDOUT_VAL_YEAR = holdout_meta["val_season_year"]

train_df = df[df["date"].isin(train_dates)].copy()
val_df = df[df["date"].isin(val_dates)].copy()
test_df = df[df["date"].isin(test_dates)].copy()

baseline_lookup = (
    train_df.groupby(["route_id", "hour", "is_weekend"])["demand"].median().rename("baseline_demand")
)
fallback = train_df.groupby(["route_id", "hour"])["demand"].median().rename("fallback_demand")

def attach_baseline(d: pd.DataFrame) -> pd.DataFrame:
    d = d.merge(baseline_lookup, on=["route_id", "hour", "is_weekend"], how="left")
    d = d.merge(fallback, on=["route_id", "hour"], how="left")
    d["baseline_demand"] = d["baseline_demand"].fillna(d["fallback_demand"])
    return d.drop(columns=["fallback_demand"])

train_df = attach_baseline(train_df)
val_df = attach_baseline(val_df)
test_df = attach_baseline(test_df)

train_df = srp.fill_lag_from_train(train_df, train_df, lag_cols=LAG_FEATURE_COLS)
val_df = srp.fill_lag_from_train(val_df, train_df, lag_cols=LAG_FEATURE_COLS)
test_df = srp.fill_lag_from_train(test_df, train_df, lag_cols=LAG_FEATURE_COLS)

for part in (train_df, val_df, test_df):
    part["log_baseline"] = np.log1p(part["baseline_demand"])
    part["residual_log_demand"] = np.log1p(part["demand"]) - part["log_baseline"]

NUM_FEATURES = srp.build_num_feature_list(
    HOURLY_FACTOR_COLS,
    use_lags=USE_LAG_FEATURES,
    lag_cols=LAG_FEATURE_COLS,
)
NUM_FEATURES = [c for c in NUM_FEATURES if c in train_df.columns]

y_train = train_df["residual_log_demand"].values
y_val = val_df["residual_log_demand"].values
y_test = test_df["residual_log_demand"].values

scaler = StandardScaler()
X_train_num = scaler.fit_transform(train_df[NUM_FEATURES])
X_val_num = scaler.transform(val_df[NUM_FEATURES])
X_test_num = scaler.transform(test_df[NUM_FEATURES])
X_train_route = train_df["route_idx"].values.reshape(-1, 1)
X_val_route = val_df["route_idx"].values.reshape(-1, 1)
X_test_route = test_df["route_idx"].values.reshape(-1, 1)

print(f"NUM_FEATURES ({len(NUM_FEATURES)}): {NUM_FEATURES}")
print(f"Hold-out: test={HOLDOUT_TEST_SEASON}_{HOLDOUT_TEST_YEAR}, val={HOLDOUT_VAL_SEASON}_{HOLDOUT_VAL_YEAR}")
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print(f"Target residual — train mean={y_train.mean():.3f}, std={y_train.std():.3f}")



NUM_FEATURES (20): ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'is_us_holiday', 'weather_label_code', 'is_rain', 'is_snow', 'is_severe_wind', 'is_peak_morning', 'is_peak_evening', 'is_overnight', 'is_major_event_window', 'log_baseline', 'log_lag_24h', 'log_lag_168h', 'log_rolling_7d']
Hold-out: test=autumn_2025, val=summer_2025
Train 299,072 | Val 50,805 | Test 49,963
Target residual — train mean=-0.019, std=0.232


## MLP training


In [4]:
def _fit_inputs(route_idx, num_feat):
    return srp.format_keras_inputs(route_idx, num_feat, use_route_embedding=USE_ROUTE_EMBEDDING)

mlp_model = srp.build_demand_model(
    len(route_to_idx),
    len(NUM_FEATURES),
    use_route_embedding=USE_ROUTE_EMBEDDING,
    hidden=NN_HIDDEN,
    dropout=NN_DROPOUT,
    use_batch_norm=USE_BATCH_NORM,
)
mlp_model.summary()

train_weights = np.ones(len(train_df), dtype=float)
peak_mask = train_df["hour"].isin(PEAK_HOURS)
train_weights[peak_mask.to_numpy()] = PEAK_SAMPLE_WEIGHT
print(f"Peak sample weight x{PEAK_SAMPLE_WEIGHT}: {peak_mask.sum():,}/{len(train_df):,}")

early = EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)
t0 = time.perf_counter()
mlp_history = mlp_model.fit(
    _fit_inputs(X_train_route, X_train_num),
    y_train,
    validation_data=(_fit_inputs(X_val_route, X_val_num), y_val),
    sample_weight=train_weights,
    epochs=NN_EPOCHS_MAIN,
    batch_size=32,
    verbose=0,
    callbacks=[early],
)
mlp_train_sec = time.perf_counter() - t0
mlp_n_params = int(mlp_model.count_params())
print(f"MLP done @ epoch {len(mlp_history.history['loss'])} | {mlp_train_sec:.1f}s | params={mlp_n_params:,}")

def predict_resid_mlp(route_idx_arr, num_feat_arr):
    return mlp_model.predict(_fit_inputs(route_idx_arr, num_feat_arr), verbose=0).ravel()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ route_idx           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ route_emb           │ (None, 1, 4)      │        100 │ route_idx[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 4)         │          0 │ route_emb[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_features        │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 24)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ num_features[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_0 (Dense)     │ (None, 64)        │      1,600 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_0                │ (None, 64)        │        256 │ dense_0[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_0 (Dropout) │ (None, 64)        │          0 │ bn_0[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dropout_0[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 32)        │        128 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_log_demand │ (None, 1)         │         33 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,197 (16.39 KB)

 Trainable params: 4,005 (15.64 KB)

 Non-trainable params: 192 (768.00 B)

Peak sample weight x2.0: 79,636/299,072


KeyboardInterrupt: 

## Blend (MLP + HistGBM)


In [ ]:
t0 = time.perf_counter()
histgbm_model = srp.fit_histgbm_demand(X_train_num, y_train, sample_weight=train_weights)
blend_train_sec = time.perf_counter() - t0
blend_n_params = getattr(histgbm_model, "n_trees_", None) or len(getattr(histgbm_model, "train_score_", []))
print(f"HistGBM fitted in {blend_train_sec:.1f}s | trees/iter proxy={blend_n_params}")

if TUNE_RESID_CLIP_ON_VAL:
    best_clip, best_mae = RESID_CLIP[1], float("inf")
    for clip_hi in RESID_CLIP_CANDIDATES:
        tmp_clip = (-clip_hi, clip_hi)
        pred_resid = np.clip(predict_resid_mlp(X_val_route, X_val_num), *tmp_clip)
        pred_val = srp.residuals_to_demand(val_df["log_baseline"].values, pred_resid, tmp_clip)
        mae_val = mean_absolute_error(val_df["demand"].values, pred_val)
        print(f"  val MAE @ clip +/-{clip_hi:.2f}: {mae_val:,.1f}")
        if mae_val < best_mae:
            best_mae, best_clip = mae_val, clip_hi
    RESID_CLIP = (-best_clip, best_clip)
    print(f"-> RESID_CLIP = +/-{best_clip:.2f} (val MAE={best_mae:,.1f})")

pred_mlp_val = np.clip(predict_resid_mlp(X_val_route, X_val_num), *RESID_CLIP)
pred_gbm_val = histgbm_model.predict(X_val_num)
blend_weight, blend_val_mae = srp.tune_blend_weight_mae(
    pred_mlp_val,
    pred_gbm_val,
    val_df["log_baseline"].values,
    val_df["demand"].values,
    RESID_CLIP,
)
print(f"Blend weight (MLP)={blend_weight:.2f} | val MAE={blend_val_mae:,.1f}")

def predict_demand_blend(route_idx_arr, num_feat_arr, log_baseline):
    pred_mlp = np.clip(predict_resid_mlp(route_idx_arr, num_feat_arr), *RESID_CLIP)
    pred_gbm = histgbm_model.predict(num_feat_arr)
    pred_resid = srp.blend_residual_predictions(pred_mlp, pred_gbm, blend_weight)
    return srp.residuals_to_demand(log_baseline, pred_resid, RESID_CLIP)



## LSTM training


In [ ]:
def attach_scaled_features(d: pd.DataFrame, scaler_obj, feature_cols: list[str], prefix: str = "scaled_") -> pd.DataFrame:
    out = d.copy()
    arr = scaler_obj.transform(out[feature_cols].values)
    for i, col in enumerate(feature_cols):
        out[f"{prefix}{col}"] = arr[:, i]
    return out

lstm_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
lstm_df = attach_scaled_features(lstm_df, scaler, NUM_FEATURES)
LSTM_FEATURE_COLS = [f"scaled_{c}" for c in NUM_FEATURES]

train_date_set = set(pd.to_datetime(train_dates))
val_date_set = set(pd.to_datetime(val_dates))
test_date_set = set(pd.to_datetime(test_dates))

X_lstm_train, y_lstm_train, meta_lstm_train, stats_train = srp.prepare_lstm_sequences(
    lstm_df,
    LSTM_SEQ_LEN,
    LSTM_FEATURE_COLS,
    "residual_log_demand",
    train_dates=train_date_set,
    val_dates=val_date_set,
    test_dates=test_date_set,
    target_split="train",
)
X_lstm_val, y_lstm_val, meta_lstm_val, stats_val = srp.prepare_lstm_sequences(
    lstm_df,
    LSTM_SEQ_LEN,
    LSTM_FEATURE_COLS,
    "residual_log_demand",
    train_dates=train_date_set,
    val_dates=val_date_set,
    test_dates=test_date_set,
    target_split="val",
)
X_lstm_test, y_lstm_test, meta_lstm_test, stats_test = srp.prepare_lstm_sequences(
    lstm_df,
    LSTM_SEQ_LEN,
    LSTM_FEATURE_COLS,
    "residual_log_demand",
    train_dates=train_date_set,
    val_dates=val_date_set,
    test_dates=test_date_set,
    target_split="test",
    meta_cols=["route_id", "date", "hour", "log_baseline", "demand"],
)
print("LSTM sequences:", stats_train, stats_val, stats_test)

lstm_model = srp.build_lstm_demand_model(
    LSTM_SEQ_LEN,
    len(LSTM_FEATURE_COLS),
    hidden_units=LSTM_HIDDEN,
    num_lstm_layers=LSTM_LAYERS,
    dropout=NN_DROPOUT,
    use_batch_norm=USE_BATCH_NORM,
)
lstm_model.summary()

# Peak weights aligned to LSTM train meta (route+date+hour)
lstm_train_keys = meta_lstm_train.merge(
    train_df.assign(_w=train_weights)[["route_id", "date", "hour", "_w"]],
    on=["route_id", "date", "hour"],
    how="left",
)
lstm_sample_weight = lstm_train_keys["_w"].fillna(1.0).values

lstm_early = EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)
t0 = time.perf_counter()
lstm_history = lstm_model.fit(
    X_lstm_train,
    y_lstm_train,
    validation_data=(X_lstm_val, y_lstm_val),
    sample_weight=lstm_sample_weight,
    epochs=LSTM_EPOCHS,
    batch_size=64,
    verbose=0,
    callbacks=[lstm_early],
)
lstm_train_sec = time.perf_counter() - t0
lstm_n_params = int(lstm_model.count_params())
print(f"LSTM done @ epoch {len(lstm_history.history['loss'])} | {lstm_train_sec:.1f}s | params={lstm_n_params:,}")



## Evaluation


In [ ]:
# Align LSTM test rows with tabular test set on route+date+hour
meta_lstm_test = meta_lstm_test.copy()
meta_lstm_test["date"] = pd.to_datetime(meta_lstm_test["date"]).dt.normalize()
meta_lstm_test["_lstm_idx"] = np.arange(len(meta_lstm_test))

test_keys = test_df[["route_id", "date", "hour"]].copy()
test_keys["date"] = pd.to_datetime(test_keys["date"]).dt.normalize()
test_keys["row_idx"] = np.arange(len(test_df))

aligned = meta_lstm_test.merge(test_keys, on=["route_id", "date", "hour"], how="inner")
print(f"LSTM test samples aligned with tabular test: {len(aligned):,} / {len(meta_lstm_test):,}")

idx = aligned["row_idx"].values
y_true_aligned = test_df.iloc[idx]["demand"].values
log_base_aligned = test_df.iloc[idx]["log_baseline"].values
baseline_aligned = test_df.iloc[idx]["baseline_demand"].values

pred_mlp_resid = np.clip(predict_resid_mlp(X_test_route[idx], X_test_num[idx]), *RESID_CLIP)
pred_mlp = srp.residuals_to_demand(log_base_aligned, pred_mlp_resid, RESID_CLIP)
pred_blend = predict_demand_blend(X_test_route[idx], X_test_num[idx], log_base_aligned)

lstm_pos = aligned["_lstm_idx"].values
pred_lstm_resid = np.clip(lstm_model.predict(X_lstm_test[lstm_pos], verbose=0).ravel(), *RESID_CLIP)
pred_lstm = srp.residuals_to_demand(log_base_aligned, pred_lstm_resid, RESID_CLIP)

m_base = srp.regression_metrics(y_true_aligned, baseline_aligned)
m_mlp = srp.regression_metrics(y_true_aligned, pred_mlp)
m_blend = srp.regression_metrics(y_true_aligned, pred_blend)
m_lstm = srp.regression_metrics(y_true_aligned, pred_lstm)

comparison_df = pd.DataFrame([
    {
        "model": "baseline",
        "mae": m_base["mae"],
        "rmse": m_base["rmse"],
        "r2": m_base["r2"],
        "mape_pct": m_base["mape_pct"],
        "smape_pct": m_base["smape_pct"],
        "train_sec": 0.0,
        "n_params": 0,
    },
    {
        "model": "mlp",
        "mae": m_mlp["mae"],
        "rmse": m_mlp["rmse"],
        "r2": m_mlp["r2"],
        "mape_pct": m_mlp["mape_pct"],
        "smape_pct": m_mlp["smape_pct"],
        "train_sec": mlp_train_sec,
        "n_params": mlp_n_params,
    },
    {
        "model": "blend",
        "mae": m_blend["mae"],
        "rmse": m_blend["rmse"],
        "r2": m_blend["r2"],
        "mape_pct": m_blend["mape_pct"],
        "smape_pct": m_blend["smape_pct"],
        "train_sec": mlp_train_sec + blend_train_sec,
        "n_params": mlp_n_params + int(blend_n_params),
    },
    {
        "model": "lstm",
        "mae": m_lstm["mae"],
        "rmse": m_lstm["rmse"],
        "r2": m_lstm["r2"],
        "mape_pct": m_lstm["mape_pct"],
        "smape_pct": m_lstm["smape_pct"],
        "train_sec": lstm_train_sec,
        "n_params": lstm_n_params,
    },
])
comparison_df.to_csv(OUT_DIR / "model_comparison.csv", index=False)
display(comparison_df.round(3))

eval_bundle = {
    "holdout_test": f"{HOLDOUT_TEST_SEASON}_{HOLDOUT_TEST_YEAR}",
    "holdout_val": f"{HOLDOUT_VAL_SEASON}_{HOLDOUT_VAL_YEAR}",
    "n_aligned_test": int(len(aligned)),
    "resid_clip": float(RESID_CLIP[1]),
    "blend_mlp_weight": float(blend_weight),
    "lstm_seq_len": LSTM_SEQ_LEN,
    "metrics": {
        "baseline": m_base,
        "mlp": m_mlp,
        "blend": m_blend,
        "lstm": m_lstm,
    },
}
with open(OUT_DIR / "model_comparison.json", "w", encoding="utf-8") as f:
    json.dump(eval_bundle, f, indent=2)



## Plots


In [ ]:
# Pred vs actual — 3 models
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, pred, name, m in zip(
    axes,
    [pred_mlp, pred_blend, pred_lstm],
    ["MLP", "Blend", "LSTM"],
    [m_mlp, m_blend, m_lstm],
):
    ax.scatter(y_true_aligned, pred, alpha=0.35, s=14)
    lim = max(y_true_aligned.max(), pred.max())
    ax.plot([0, lim], [0, lim], "r--", lw=1)
    ax.set_title(f"{name} | MAE={m['mae']:,.0f} R2={m['r2']:.3f}")
    ax.set_xlabel("Actual demand")
    ax.set_ylabel("Predicted demand")
plt.suptitle(f"Hold-out {HOLDOUT_TEST_SEASON}_{HOLDOUT_TEST_YEAR} (aligned n={len(aligned):,})")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_pred_vs_actual.png", dpi=120, bbox_inches="tight")
plt.show()

# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(mlp_history.history["loss"], label="train")
if "val_loss" in mlp_history.history:
    axes[0].plot(mlp_history.history["val_loss"], label="val")
axes[0].set_title("MLP loss")
axes[0].legend()
axes[1].plot(lstm_history.history["loss"], label="train")
if "val_loss" in lstm_history.history:
    axes[1].plot(lstm_history.history["val_loss"], label="val")
axes[1].set_title(f"LSTM loss (seq={LSTM_SEQ_LEN})")
axes[1].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_loss_curves.png", dpi=120, bbox_inches="tight")
plt.show()

# Hourly MAE
hourly_eval = pd.DataFrame({
    "hour": test_df.iloc[idx]["hour"].values,
    "y_true": y_true_aligned,
    "mlp": pred_mlp,
    "blend": pred_blend,
    "lstm": pred_lstm,
})
hourly_mae = hourly_eval.groupby("hour").apply(
    lambda g: pd.Series({
        "mlp": mean_absolute_error(g["y_true"], g["mlp"]),
        "blend": mean_absolute_error(g["y_true"], g["blend"]),
        "lstm": mean_absolute_error(g["y_true"], g["lstm"]),
    }),
    include_groups=False,
).reset_index()
hourly_mae.to_csv(OUT_DIR / "hourly_mae.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly_mae["hour"], hourly_mae["mlp"], marker="o", label="MLP")
ax.plot(hourly_mae["hour"], hourly_mae["blend"], marker="s", label="Blend")
ax.plot(hourly_mae["hour"], hourly_mae["lstm"], marker="^", label="LSTM")
ax.set_xlabel("Hour")
ax.set_ylabel("MAE")
ax.set_title("Hourly MAE on aligned hold-out test")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_hourly_mae.png", dpi=120, bbox_inches="tight")
plt.show()

# Route MAE
route_eval = pd.DataFrame({
    "route_id": test_df.iloc[idx]["route_id"].values,
    "y_true": y_true_aligned,
    "mlp": pred_mlp,
    "blend": pred_blend,
    "lstm": pred_lstm,
})
route_mae = route_eval.groupby("route_id").apply(
    lambda g: pd.Series({
        "mlp": mean_absolute_error(g["y_true"], g["mlp"]),
        "blend": mean_absolute_error(g["y_true"], g["blend"]),
        "lstm": mean_absolute_error(g["y_true"], g["lstm"]),
    }),
    include_groups=False,
).reset_index().sort_values("blend")
route_mae.to_csv(OUT_DIR / "route_mae.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(route_mae))
w = 0.25
ax.bar(x - w, route_mae["mlp"], width=w, label="MLP")
ax.bar(x, route_mae["blend"], width=w, label="Blend")
ax.bar(x + w, route_mae["lstm"], width=w, label="LSTM")
ax.set_xticks(x)
ax.set_xticklabels(route_mae["route_id"], rotation=45, ha="right")
ax.set_ylabel("MAE")
ax.set_title("Route-level MAE (aligned hold-out test)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_route_mae.png", dpi=120, bbox_inches="tight")
plt.show()



## Conclusion

Ba mô hình dự báo **residual log-demand** trên cùng tập test mùa **autumn_2025** (các hàng LSTM được align với MLP/Blend theo `route_id` + `date` + `hour`).

**Khuyến nghị pipeline chính** (cập nhật sau khi chạy cell summary):

- **Blend MLP+HistGBM** — lựa chọn production hiện tại: thường tốt nhất hoặc ngang MLP, thêm HistGBM bắt pattern phi tuyến mà không cần sequence dài.
- **MLP** — khi cần inference đơn giản, ít phụ thuộc tree ensemble; train nhanh hơn LSTM.
- **LSTM** — chỉ đáng cân nhắc nếu vượt Blend trên aligned test; nếu kém hơn, xem `stats_test` (boundary drops) và MAE theo giờ/route — thường do thiếu lịch sử 24h đầu block test hoặc route ít dữ liệu.

Output: `outputs/model_comparison/model_comparison.csv`, JSON metrics, và các figure PNG.


In [ ]:
best = comparison_df.loc[comparison_df["model"] != "baseline"].sort_values("mae").iloc[0]
print("=== Model comparison summary ===")
print(f"Hold-out test: {HOLDOUT_TEST_SEASON}_{HOLDOUT_TEST_YEAR} | val tuned: {HOLDOUT_VAL_SEASON}_{HOLDOUT_VAL_YEAR}")
print(f"Aligned test rows: {len(aligned):,} | RESID_CLIP=+/-{RESID_CLIP[1]:.2f} | blend MLP weight={blend_weight:.2f}")
print()
print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:,.3f}"))
print()
print(f"LSTM sequence stats (test): {stats_test}")
print(f"Best model by MAE: {best['model']} (MAE={best['mae']:,.1f}, SMAPE={best['smape_pct']:.1f}%, train_sec={best['train_sec']:,.1f}s, n_params={int(best['n_params']):,})")

